# Notebook 01: RetailMart 360 — Data Understanding & Relational Audit

**Project:** RetailMart 360 — Retail Sales & Customer Intelligence  
**Role:** Senior Data Analyst & Analytics Consultant  
**Environment:** Python 3.13 | Pandas | Relational Schema Inspection  

---

## 1. Business Objective & Analytical Questions
The objective of this notebook is to establish an empirical understanding of the RetailMart transaction database. As a retail analytics consultant, we must verify schema structure, entity relationships, chronological coverage, data types, missing values, and data integrity before performing any business analysis.

### Core Questions to Answer:
1. What tables exist in the retail ecosystem and what are their primary/foreign keys?
2. What are the dimensionalities (row and column counts) of each entity?
3. What is the chronological window of transactional data?
4. What missing values, anomalies, or potential pitfalls exist in the raw extracts?
5. How do orders, line items, customers, and payments relate at different business grains?


In [1]:
import os
import glob
import pandas as pd
import numpy as np

# Define relative data path
RAW_DIR = "../data/raw"
raw_files = sorted(glob.glob(os.path.join(RAW_DIR, "*.csv")))
print(f"Found {len(raw_files)} raw CSV tables in {RAW_DIR}:")
for f in raw_files:
    print(f"  - {os.path.basename(f)}")


Found 8 raw CSV tables in ../data/raw:
  - customers.csv
  - employees.csv
  - offices.csv
  - orderdetails.csv
  - orders.csv
  - payments.csv
  - productlines.csv
  - products.csv


## 2. Dataset Inventory & Schema Dimensionality
We load each table and inspect row counts, column counts, and memory usage.


In [2]:
inventory = []
raw_dfs = {}
for path in raw_files:
    tbl_name = os.path.splitext(os.path.basename(path))[0]
    df = pd.read_csv(path)
    raw_dfs[tbl_name] = df
    inventory.append({
        "Table": tbl_name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Memory (KB)": round(df.memory_usage(deep=True).sum() / 1024, 2)
    })

inv_df = pd.DataFrame(inventory).sort_values("Rows", ascending=False)
display(inv_df)


Table  Rows  Columns  Memory (KB)
3  orderdetails  2996        5       117.16
4        orders   326        7        54.34
5      payments   273        4        27.91
0     customers   122       13        56.77
7      products   110        9        66.19
1     employees    23        8         8.42
2       offices     7        9         3.18
6  productlines     7        4         3.94


## 3. Date Span & Transactional Boundaries
We inspect the minimum and maximum recorded dates across order placement, required fulfillment dates, shipping dates, and customer payments.


In [3]:
orders_df = raw_dfs["orders"].copy()
payments_df = raw_dfs["payments"].copy()

print("Orders Date Coverage:")
print(f"  Order Date Span:    {orders_df['orderDate'].min()} to {orders_df['orderDate'].max()}")
print(f"  Required Date Span: {orders_df['requiredDate'].min()} to {orders_df['requiredDate'].max()}")
print(f"  Shipped Date Span:  {orders_df['shippedDate'].dropna().min()} to {orders_df['shippedDate'].dropna().max()}")
print(f"  Unshipped Orders:   {orders_df['shippedDate'].isna().sum()} orders without shipped date")

print("\nPayments Date Coverage:")
print(f"  Payment Date Span:  {payments_df['paymentDate'].min()} to {payments_df['paymentDate'].max()}")


Orders Date Coverage:
  Order Date Span:    2003-01-06 to 2005-05-31
  Required Date Span: 2003-01-13 to 2005-06-11
  Shipped Date Span:  2003-01-10 to 2005-05-20
  Unshipped Orders:   14 orders without shipped date

Payments Date Coverage:
  Payment Date Span:  2003-01-16 to 2005-06-09


## 4. Missing Values & Ingestion Pitfalls Audit
We examine columns with null values to determine whether missingness is legitimate business absence or an ETL defect.


In [4]:
for name, df in raw_dfs.items():
    nulls = df.isna().sum()
    null_cols = nulls[nulls > 0]
    if len(null_cols) > 0:
        print(f"Table '{name}':")
        for col, cnt in null_cols.items():
            pct = round(cnt / len(df) * 100, 2)
            print(f"  - {col}: {cnt} nulls ({pct}%)")


Table 'customers':
  - addressLine2: 100 nulls (81.97%)
  - state: 73 nulls (59.84%)
  - postalCode: 7 nulls (5.74%)
  - salesRepEmployeeNumber: 22 nulls (18.03%)
Table 'employees':
  - reportsTo: 1 nulls (4.35%)
Table 'offices':
  - addressLine2: 2 nulls (28.57%)
  - state: 3 nulls (42.86%)
  - territory: 3 nulls (42.86%)
Table 'orders':
  - shippedDate: 14 nulls (4.29%)
  - comments: 246 nulls (75.46%)
Table 'productlines':
  - htmlDescription: 7 nulls (100.0%)
  - image: 7 nulls (100.0%)


### Key Findings & Analytical Interpretations:
1. **Offices Territory Ingestion Defect:** In `offices.csv`, the territory code `'NA'` (North America) was parsed by pandas as `NaN`. In our cleaning pipeline, this must be explicitly corrected to `'NA'`.
2. **Unassigned Customers:** 22 customer accounts have null `salesRepEmployeeNumber`. None of these 22 accounts have ever placed an order (dormant prospect accounts).
3. **CEO Hierarchy Root:** In `employees`, Diane Murphy (President, 1002) legitimately has a null `reportsTo`.
4. **Unshipped Orders:** 14 orders have a null `shippedDate` because they are In Process (6), On Hold (4), or Cancelled (4).
5. **Truncated 2005 Horizon:** Transactional data terminates on May 31, 2005. Full-year aggregations for 2005 would create a misleading comparison against full 12-month periods in 2003 and 2004.
